# LABORATORIO N. 09

## Semana 9 - Administracion y Almacenamiento de Base de Datos

**Curso:** Fundamentos de Gestion de Datos  
**Herramientas:** Google Colab, SQLite, Python, pandas  
**Base real:** SQLite descargada/subida desde GitHub  
**Repositorio:** https://github.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos

### Alineacion con el silabo

Este laboratorio desarrolla:

- Arquitectura DBMS con SQLite.
- DML: `INSERT`, `UPDATE`, `DELETE`.
- Transacciones: `BEGIN`, `COMMIT`, `ROLLBACK`.
- DCL conceptual: `GRANT`, `REVOKE`, roles y permisos.
- Almacenamiento local, nube e hibrido.

**Nota importante:** el laboratorio ahora detecta la tabla real disponible. Si tu base tiene `ventas_original`, trabajara con esa tabla. Si tiene `ventas`, trabajara con `ventas`. No exige tablas que tu base no tiene.

## Actividad 1 - Conceptos previos

| Concepto | Respuesta |
|---|---|
| DML | |
| INSERT | |
| UPDATE | |
| DELETE | |
| Transaccion | |
| COMMIT | |
| ROLLBACK | |
| DCL | |
| GRANT | |
| REVOKE | |
| Almacenamiento local | |
| Almacenamiento en nube | |
| Almacenamiento hibrido | |

In [ ]:
# Paso 1: librerias y configuracion para Colab

import shutil
import sqlite3
import urllib.request
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)

DB_URL = "https://raw.githubusercontent.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos/main/casos/16_abarrotes_ventas_inventario/abarrotes_ventas_inventario.db"
DB_ORIGINAL = Path("abarrotes_ventas_inventario.db")
DB_TRABAJO = Path("trabajo_lab_d9.db")


def mostrar_limpio(df, reemplazo=""):
    """Devuelve una copia para visualizar sin NaN/None en las salidas."""
    salida = df.copy()
    return salida.where(pd.notna(salida), reemplazo)


def texto_limpio(serie, reemplazo="SIN_DATO"):
    """Normaliza columnas de texto sin convertir faltantes en la palabra 'nan'."""
    return serie.astype("string").fillna(reemplazo).astype(str).replace({"nan": reemplazo, "None": reemplazo, "": reemplazo})

print("Entorno listo")



In [ ]:
# Paso 2: descargar la base desde GitHub o subirla manualmente

def preparar_base():
    if not DB_ORIGINAL.exists() or DB_ORIGINAL.stat().st_size <= 1000:
        try:
            print("Descargando base SQLite desde GitHub...")
            urllib.request.urlretrieve(DB_URL, DB_ORIGINAL)
            if DB_ORIGINAL.stat().st_size <= 1000:
                raise ValueError("El archivo descargado no parece una base SQLite valida.")
            print("Descarga correcta:", DB_ORIGINAL.resolve())
        except Exception as error:
            print("No se pudo descargar automaticamente.")
            print("Detalle:", error)
            print("Sube manualmente el archivo .db que estas usando en el curso.")
            from google.colab import files
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No se subio ningun archivo.")
            nombre = next(iter(uploaded.keys()))
            shutil.move(nombre, DB_ORIGINAL)

    shutil.copy2(DB_ORIGINAL, DB_TRABAJO)
    print("Copia de trabajo:", DB_TRABAJO.resolve())
    print("Tamano:", DB_TRABAJO.stat().st_size, "bytes")


preparar_base()

In [ ]:
# Paso 3: conectar e inspeccionar tablas reales

conn = sqlite3.connect(DB_TRABAJO)
conn.isolation_level = None
cur = conn.cursor()

tablas = pd.read_sql_query(
    '''
    SELECT name AS tabla
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    ''',
    conn
)

display(mostrar_limpio(tablas))

if tablas.empty:
    raise ValueError("La base no contiene tablas.")

# Detectar tabla base real.
preferencias = ["ventas_original", "ventas", "dataset_abarrotes_limpio"]
tablas_disponibles = tablas["tabla"].tolist()

TABLA_BASE = None
for candidata in preferencias:
    if candidata in tablas_disponibles:
        TABLA_BASE = candidata
        break

if TABLA_BASE is None:
    TABLA_BASE = tablas_disponibles[0]

print("Tabla base seleccionada:", TABLA_BASE)

estructura = pd.read_sql_query(f"PRAGMA table_info('{TABLA_BASE}')", conn)
estructura["dflt_value"] = estructura["dflt_value"].fillna("Sin valor por defecto")
display(mostrar_limpio(estructura))

vista_original = pd.read_sql_query(f"SELECT * FROM '{TABLA_BASE}' LIMIT 5", conn)
display(mostrar_limpio(vista_original, "SIN_DATO"))

faltantes_original = (
    pd.read_sql_query(f"SELECT * FROM '{TABLA_BASE}'", conn)
    .isna()
    .sum()
    .reset_index()
)
faltantes_original.columns = ["columna", "valores_faltantes"]
faltantes_original = faltantes_original[faltantes_original["valores_faltantes"] > 0]
print("Resumen de valores faltantes en la tabla original:")
if faltantes_original.empty:
    print("No se detectaron valores faltantes en la tabla original.")
else:
    display(faltantes_original)



### Pregunta 1

Que tabla real detecto el notebook? Por que eso evita trabajar con una base ficticia?

**Respuesta:** ____________________________________________________________________

In [ ]:
# Paso 4: crear tabla de trabajo estandarizada desde datos reales

df_base = pd.read_sql_query(f"SELECT * FROM '{TABLA_BASE}'", conn)

def buscar_columna(posibles):
    columnas = list(df_base.columns)
    columnas_lower = {c.lower(): c for c in columnas}
    for posible in posibles:
        posible = posible.lower()
        if posible in columnas_lower:
            return columnas_lower[posible]
    for c in columnas:
        c_lower = c.lower()
        if any(posible.lower() in c_lower for posible in posibles):
            return c
    return None

def limpiar_numero(serie):
    return (
        serie.astype(str)
        .str.replace("S/", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.extract(r"([-+]?[0-9]*\.?[0-9]+)", expand=False)
        .astype(float)
    )

col_id = buscar_columna(["id_venta", "id", "codigo"])
col_fecha = buscar_columna(["fecha_operacion", "fecha", "date"])
col_producto = buscar_columna(["productos_nombre", "producto", "nombre_producto", "item"])
col_cantidad = buscar_columna(["cantidad_unidades", "cantidad", "unidades"])
col_monto = buscar_columna(["monto_venta_soles", "monto", "total", "venta"])

df_lab = pd.DataFrame()
df_lab["id_operacion"] = range(1, len(df_base) + 1)
df_lab["fuente_tabla"] = TABLA_BASE
df_lab["id_origen"] = texto_limpio(df_base[col_id]) if col_id else df_lab["id_operacion"].astype(str)
df_lab["fecha_operacion"] = texto_limpio(df_base[col_fecha]) if col_fecha else "SIN_DATO"
df_lab["producto"] = texto_limpio(df_base[col_producto]) if col_producto else "registro_real"
df_lab["cantidad"] = limpiar_numero(df_base[col_cantidad]).fillna(0) if col_cantidad else 0
df_lab["monto"] = limpiar_numero(df_base[col_monto]).fillna(0) if col_monto else 0
df_lab["estado"] = "ORIGINAL"
df_lab["observacion"] = "BASE_REAL"

cur.execute("DROP TABLE IF EXISTS ventas_lab_d9")
df_lab.to_sql("ventas_lab_d9", conn, if_exists="replace", index=False)

print("Columnas detectadas:")
print("ID:", col_id, "| Fecha:", col_fecha, "| Producto:", col_producto, "| Cantidad:", col_cantidad, "| Monto:", col_monto)
print("Tabla de trabajo creada: ventas_lab_d9")
display(mostrar_limpio(pd.read_sql_query("SELECT * FROM ventas_lab_d9 LIMIT 10", conn), "SIN_DATO"))



## Actividad 2 - DML con transacciones

Las operaciones se hacen sobre `ventas_lab_d9`, una tabla de trabajo creada a partir de los registros reales detectados en la base. Esto protege la tabla original y permite practicar DML sin dañar datos fuente.

In [ ]:
# Paso 5: evidencia inicial

display(mostrar_limpio(pd.read_sql_query("SELECT COUNT(*) AS registros_iniciales FROM ventas_lab_d9", conn), "SIN_DATO"))
display(mostrar_limpio(pd.read_sql_query("SELECT estado, observacion, COUNT(*) AS registros FROM ventas_lab_d9 GROUP BY estado, observacion", conn), "SIN_DATO"))


In [ ]:
# Paso 6: INSERT de 10 registros basados en registros reales

base_insert = pd.read_sql_query(
    "SELECT fecha_operacion, producto, cantidad, monto FROM ventas_lab_d9 WHERE estado = 'ORIGINAL' LIMIT 10",
    conn
)

max_id = int(pd.read_sql_query("SELECT MAX(id_operacion) AS max_id FROM ventas_lab_d9", conn).iloc[0]["max_id"])

nuevos = []
for i, row in base_insert.iterrows():
    nuevos.append((
        max_id + i + 1,
        TABLA_BASE,
        f"COPIA_REAL_{i+1}",
        row["fecha_operacion"],
        row["producto"],
        float(row["cantidad"]),
        float(row["monto"]),
        "INSERTADO",
        "LAB_D9_INSERT"
    ))

try:
    cur.execute("BEGIN")
    cur.executemany(
        '''
        INSERT INTO ventas_lab_d9 (
            id_operacion, fuente_tabla, id_origen, fecha_operacion,
            producto, cantidad, monto, estado, observacion
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        nuevos
    )
    cur.execute("COMMIT")
    print("INSERT confirmado:", len(nuevos), "registros")
except Exception as error:
    cur.execute("ROLLBACK")
    raise RuntimeError(f"Error en INSERT: {error}")

display(mostrar_limpio(pd.read_sql_query("SELECT * FROM ventas_lab_d9 WHERE observacion = 'LAB_D9_INSERT'", conn), "SIN_DATO"))


### Pregunta 2

Por que el INSERT se hace dentro de una transaccion?

**Respuesta:** ____________________________________________________________________

In [ ]:
# Paso 7: UPDATE de 3 registros

ids_update = pd.read_sql_query(
    "SELECT id_operacion FROM ventas_lab_d9 WHERE observacion = 'LAB_D9_INSERT' ORDER BY id_operacion LIMIT 3",
    conn
)["id_operacion"].tolist()

display(mostrar_limpio(pd.read_sql_query(
    "SELECT * FROM ventas_lab_d9 WHERE id_operacion IN (?, ?, ?)",
    conn,
    params=ids_update
), "SIN_DATO"))

try:
    cur.execute("BEGIN")
    cur.executemany(
        '''
        UPDATE ventas_lab_d9
        SET estado = ?,
            observacion = ?,
            monto = ROUND(monto * 1.05, 2)
        WHERE id_operacion = ?
        ''',
        [("ACTUALIZADO", "LAB_D9_UPDATE", int(i)) for i in ids_update]
    )
    cur.execute("COMMIT")
    print("UPDATE confirmado:", len(ids_update), "registros")
except Exception as error:
    cur.execute("ROLLBACK")
    raise RuntimeError(f"Error en UPDATE: {error}")

display(mostrar_limpio(pd.read_sql_query(
    "SELECT * FROM ventas_lab_d9 WHERE id_operacion IN (?, ?, ?)",
    conn,
    params=ids_update
), "SIN_DATO"))


### Pregunta 3

Que riesgo existe al ejecutar un UPDATE sin WHERE?

**Respuesta:** ____________________________________________________________________

In [ ]:
# Paso 8: DELETE de 2 registros con verificacion previa

ids_delete = pd.read_sql_query(
    "SELECT id_operacion FROM ventas_lab_d9 WHERE observacion = 'LAB_D9_INSERT' ORDER BY id_operacion DESC LIMIT 2",
    conn
)["id_operacion"].tolist()

print("Registros a eliminar:", ids_delete)
display(mostrar_limpio(pd.read_sql_query(
    "SELECT * FROM ventas_lab_d9 WHERE id_operacion IN (?, ?)",
    conn,
    params=ids_delete
), "SIN_DATO"))

try:
    cur.execute("BEGIN")
    cur.executemany(
        "DELETE FROM ventas_lab_d9 WHERE id_operacion = ?",
        [(int(i),) for i in ids_delete]
    )
    cur.execute("COMMIT")
    print("DELETE confirmado:", len(ids_delete), "registros")
except Exception as error:
    cur.execute("ROLLBACK")
    raise RuntimeError(f"Error en DELETE: {error}")

display(mostrar_limpio(pd.read_sql_query(
    "SELECT * FROM ventas_lab_d9 WHERE id_operacion IN (?, ?)",
    conn,
    params=ids_delete
), "SIN_DATO"))


### Pregunta 4

Por que se debe verificar con SELECT antes de ejecutar DELETE?

**Respuesta:** ____________________________________________________________________

In [ ]:
# Paso 9: demostracion de ROLLBACK

max_id = int(pd.read_sql_query("SELECT MAX(id_operacion) AS max_id FROM ventas_lab_d9", conn).iloc[0]["max_id"])
conteo_antes = int(pd.read_sql_query("SELECT COUNT(*) AS total FROM ventas_lab_d9", conn).iloc[0]["total"])

try:
    cur.execute("BEGIN")
    cur.execute(
        '''
        INSERT INTO ventas_lab_d9 (
            id_operacion, fuente_tabla, id_origen, fecha_operacion,
            producto, cantidad, monto, estado, observacion
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        (max_id + 1, TABLA_BASE, "ROLLBACK_TEST", "2026-06-30", "registro_prueba", 1, 10, "TEMPORAL", "LAB_D9_ROLLBACK")
    )
    print("Registro temporal insertado dentro de la transaccion.")
    raise Exception("Error simulado antes de COMMIT")
    cur.execute("COMMIT")
except Exception as error:
    cur.execute("ROLLBACK")
    print("ROLLBACK ejecutado:", error)

conteo_despues = int(pd.read_sql_query("SELECT COUNT(*) AS total FROM ventas_lab_d9", conn).iloc[0]["total"])
print("Conteo antes:", conteo_antes, "| Conteo despues:", conteo_despues)
display(mostrar_limpio(pd.read_sql_query("SELECT * FROM ventas_lab_d9 WHERE observacion = 'LAB_D9_ROLLBACK'", conn), "SIN_DATO"))


### Pregunta 5

El registro temporal quedo guardado? Explica que hizo ROLLBACK.

**Respuesta:** ____________________________________________________________________

## Actividad 3 - DCL conceptual

SQLite no implementa `GRANT` y `REVOKE` como motores cliente-servidor. Aun asi, el concepto de control de acceso es parte del silabo.

In [ ]:
# Paso 10: matriz conceptual de roles

permisos = pd.DataFrame([
    ["analista_ventas", "SELECT", "No modifica registros"],
    ["operador_tienda", "SELECT, INSERT, UPDATE", "No elimina registros"],
    ["auditor", "SELECT", "Solo revision"],
    ["admin_bd", "ALL PRIVILEGES", "Uso restringido"]
], columns=["rol", "permisos", "restriccion"])

display(permisos)

print("Ejemplos conceptuales:")
print("GRANT SELECT ON ventas TO analista_ventas;")
print("GRANT INSERT, UPDATE ON ventas TO operador_tienda;")
print("REVOKE DELETE ON ventas FROM operador_tienda;")


### Pregunta 6

Que riesgo existe si todos los usuarios pueden eliminar registros?

**Respuesta:** ____________________________________________________________________

## Actividad 4 - Almacenamiento local, nube e hibrido

In [ ]:
# Paso 11: comparacion de almacenamiento

almacenamiento = pd.DataFrame([
    ["Local", "SQLite en Colab/PC", "Rapido y simple", "Se pierde si no se respalda"],
    ["Nube", "GitHub, Drive, Cloud SQL", "Colaborativo y respaldado", "Requiere permisos"],
    ["Hibrido", "SQLite local + respaldo cloud", "Flexible", "Puede duplicar versiones"],
], columns=["tipo", "ejemplo", "ventaja", "riesgo"])

display(almacenamiento)


In [ ]:
# Paso 12: evidencia final

display(mostrar_limpio(pd.read_sql_query(
    '''
    SELECT estado, observacion, COUNT(*) AS registros
    FROM ventas_lab_d9
    GROUP BY estado, observacion
    ORDER BY observacion, estado;
    ''',
    conn
), "SIN_DATO"))

display(mostrar_limpio(pd.read_sql_query("SELECT COUNT(*) AS registros_rollback FROM ventas_lab_d9 WHERE observacion = 'LAB_D9_ROLLBACK'", conn), "SIN_DATO"))
display(mostrar_limpio(pd.read_sql_query("SELECT COUNT(*) AS total_final FROM ventas_lab_d9", conn), "SIN_DATO"))

conn.close()
print("Conexion cerrada correctamente")


## Conclusiones

1. ____________________________________________________________________
2. ____________________________________________________________________
3. ____________________________________________________________________

## Entregable

- Notebook ejecutado en Google Colab.
- Evidencias antes y despues de cada DML.
- Respuestas a las preguntas.
- Base `trabajo_lab_d9.db` descargada o respaldada.